# Non stationary wormholes

In [ ]:
%load_ext autoreload
%autoreload 2

from gravityp import *
import numpy as np
from matplotlib import pyplot as plt
from scipy import optimize
import warnings

In [ ]:
plt.style.use("paper.mplstyle")
pt = 1./72.27 # Hundreds of years of history... 72.27 points to an inch.
jour_sizes = {"PRD": {"onecol": 246.*pt, "twocol": 510.*pt},
              "CQG": {"onecol": 374.*pt}, # CQG is only one column
              # Add more journals below. Can add more properties to each journal
             }
my_width = jour_sizes["PRD"]["twocol"]
golden = (1 + 5**0.5)/2 # Our figure's aspect ratio
figsize = (my_width, my_width/golden)

## Simpson-Visser

### Metric functions and potential

We consider a non stationary Simpson-Visser wormhole, whose line element reads:
\begin{equation*}
    ds^2 = -e^{2\nu}dt^2+e^{2\lambda}dr^2+e^{2\xi}d\Omega^2
\end{equation*}
where the $g_{tt}$ and $g_{rr}$ functions only depend on the radial coordinate $r$
\begin{equation*}
    e^{2\nu(r)}=e^{-2\lambda(r)}=1-\frac{2M}{\sqrt{r^2+{r_{\mathrm{throat}}}^2}}
\end{equation*}
and the areal coordinate $e^\xi$ is **time dependent**
\begin{equation*}
    e^{\xi(t,r)} = \sqrt{r^2+{r_{\mathrm{throat}}}^2} + \Xi_{02}(t) .
\end{equation*}

Hereafter, we will lways work on units of $M=1$.

In [ ]:
def f_SV(r: float, r_throat: float) -> float:
    """Radial function e^{2nu} of the Simpson-Visser metric in units of M=1"""
    return 1 - 2./np.sqrt(r**2+r_throat**2)

def areal_radius2(r: float, r_throat: float, xi_t: float) -> float:
    """areal radius squared e^{2xi} of a wormhole in units of M=1"""
    return ( np.sqrt(r**2+r_throat**2) + xi_t )**2

def compute_inner_edge_SV(r_throat):
    """Inner edge for SV metric, for which g_tt(r)=0"""
    func = lambda x: np.sqrt(4.-x**2)
    return np.piecewise(r_throat, [r_throat<=2.,r_throat>2.] , [func, 0] )

def compute_photon_rings_SV(r_throat, xi_t):
    """Compute the coordinate r of the photon ring for the SV metric"""
    # We define x^2 = r^2 + r_throat^2
    p_ring_x = np.sqrt( 9/2 + xi_t + 3*np.sqrt(9/4+xi_t) ) # Value of x at the photon ring
    func = lambda r: np.sqrt( p_ring_x**2 - r**2 ) # Position of the photon ring for the r coordinate
    return [ np.piecewise(r_throat, [ r_throat<=p_ring_x , r_throat>p_ring_x ] , [func, 0] ) ]

def compute_b_crits_SV(xi_t):
    """Compute the critical impact paremeter of a non stationary SV wormhole"""
    radical = np.sqrt(9+4*xi_t)
    return [ 0.5 * np.sqrt( 18*(3+radical)+4*xi_t*( 9+xi_t+2*radical ) ) ]

def get_SV_kwargs(r_throat, xi_t):
    # TODO: Dar la posibilidad de que radial_fun y areal puedan no tener parámetros
    return {
    'r_phs': compute_photon_rings_SV(r_throat, xi_t),
    'inner_edge': compute_inner_edge_SV(r_throat),
    'radial_fun': f_SV,
    'radial_params': (r_throat,),
    'areal': areal_radius2,
    'areal_params': (r_throat, xi_t)
    }

First we take a look to the metric function $e^{2\nu}$:

In [ ]:
params = {r'$r_\mathrm{throat}/M$' : [9,4,2,4/3,1,0]}
r_range = np.linspace(0,10,200)

with warnings.catch_warnings(action='ignore'):
    make_metric_function_plot(f_SV, params, r_range, figsize=(my_width,my_width/golden), savepath=None)

For values $r_\mathrm{throat} \gt 2M $, the horizon disappears and we have a traversable wormhole.

In [ ]:
def plot_potential_SV(ax, rr, r_throat, xi_t, label, linestyle):
    kwargs_SV = {
        'radial_fun': f_SV,
        'radial_params': (r_throat,),
        'areal': areal_radius2,
        'areal_params': (r_throat, xi_t)
    }
    ax.plot(rr, potential(rr, **kwargs_SV), label=label, linestyle=linestyle)

def make_potential_plot_SV(rr, r_throats, xi_ts, linestyles,
                        figsize=(7,7), savepath=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize, sharey=True)

    for i, xi_t in enumerate(xi_ts):
        label = r'$\Xi/M=$'+f' {xi_t}'
        plot_potential_SV(ax1, rr, r_throats[0], xi_t, label, linestyles[i])
        plot_potential_SV(ax2, rr, r_throats[1], xi_t, None, linestyles[i])

    ax1.set_ylabel(r'$V(r)/M^2$')
    ax1.text(-12, 0.08, s=r'$r_{\mathrm{throat}}=1.5M$')
    ax2.text(-12, 0.08, s=r'$r_{\mathrm{throat}}=2.5M$')
    for ax in [ax1,ax2]:
        ax.set_xlim( int(rr.min()) , int(rr.max()) )
        ax.set_ylim(0,0.1)
        ax.set_xlabel(r'$r/M$')

    handles, labels = ax1.get_legend_handles_labels()

    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.53, 1.1),
               ncol=4, frameon=True)
    plt.tight_layout()
    if savepath is not None:
        plt.savefig(savepath, format='pdf')
    plt.show()

In [ ]:
width = my_width
figsize = (width,width/3)
savepath=None
# savepath=f'latex/figures/SV/SV_potential.pdf'

rr = np.linspace(-14,14,200)
r_throats = [3/2,5/2,]
xi_ts = [-1,0,1]
linestyles = [ '--' , '-.' , '-' ]


make_potential_plot_SV(rr, r_throats, xi_ts, linestyles,
                       figsize=figsize, savepath=savepath)

2 column version of the potential plot

In [ ]:
import matplotlib.lines as mlines
from matplotlib.legend_handler import HandlerBase

class VerticalStackHandler(HandlerBase):
    """Custom legend handler that stacks elements vertically"""
    # NOTE: function made bu Gemini 3.5 Flash
    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
            line_top, line_bottom = orig_handle
            
            # Calculate the vertical midpoint relative to the text baseline
            midpoint = (height - ydescent) / 2
            # Adjust these offsets (e.g., 3 and -3 points) to change the vertical gap between the stacked lines
            y_top = midpoint + 3.0
            y_bottom = midpoint - 3.0

            legline_top = plt.Line2D(
                [0, width], [y_top, y_top],
                linestyle=line_top.get_linestyle(), color=line_top.get_color(), linewidth=line_top.get_linewidth()
            )
            legline_bottom = plt.Line2D(
                [0, width], [y_bottom, y_bottom],
                linestyle=line_bottom.get_linestyle(), color=line_bottom.get_color(), linewidth=line_bottom.get_linewidth()
            )
            
            legline_top.set_transform(trans)
            legline_bottom.set_transform(trans)
            
            return [legline_top, legline_bottom]

def plot_potential_SV(ax, rr, r_throat, xi_t, linestyle, color):
    kwargs_SV = {
        'radial_fun': f_SV,
        'radial_params': (r_throat,),
        'areal': areal_radius2,
        'areal_params': (r_throat, xi_t)
    }
    line, = ax.plot(rr, potential(rr, **kwargs_SV), linestyle=linestyle, color=color)
    return line

def make_potential_plot_SV(rr, r_throats, xi_ts, linestyles, colors,
                           figsize=(7,7), savepath=None):
    fig, ax = plt.subplots(figsize=figsize)

    lines = []
    for i, r_throat in enumerate(r_throats):
        lines.append([])
        for j, xi_t in enumerate(xi_ts):
            # label = rf'$\Xi/M={xi_t}$' if i==0 else None
            # label, color, linestyles
            lines[i].append(
                plot_potential_SV(ax, rr, r_throat, xi_t, linestyles[i], colors[j])
            )

    # plot_potential_SV(ax, rr, 0, 0, None, 'dotted', 'black') # Add Schwarzschild
    ax.set_ylabel(r'$V(r)/M^2$')
    ax.set_xlabel(r'$r/M$')
    ax.set_xlim( int(rr.min()) , int(rr.max()) )
    ax.set_ylim(0,0.1)
    
    legend_labels = [ rf'$\Xi/M = {xi_t}$' for xi_t in xi_ts ]
    legend_handles = [ (lines[0][j], lines[1][j]) for j, _ in enumerate(xi_ts) ]

    # Assign VerticalStackHandler to handle the tuple styling
    ax.legend(
            handles=legend_handles,
            labels=legend_labels,
            handler_map={tuple: VerticalStackHandler()},
            handleheight=1.0,   # Squeezes the inner drawing box height
            labelspacing=0.35,   # Restores the original, tight vertical row spacing
            handletextpad=0.5    # Padding between the line icons and the text label
        )

    if savepath is not None:
        plt.savefig(savepath, format='pdf')
    plt.show()

In [ ]:
width = my_width*0.48
figsize = (width,width/golden)
# savepath=f'latex/figures/SV/SV_potential.pdf'
savepath=None

rr = np.linspace(-10,20,200)
r_throats = [3/2,5/2,]
xi_ts = [-1,0,1]
linestyles = ['-' , '--']
colors = ['tab:blue', 'tab:orange', 'tab:green']


make_potential_plot_SV(rr, r_throats, xi_ts, linestyles, colors,
                       figsize=figsize, savepath=savepath)

### Ray tracing

In [ ]:
xi_t = 0
r_throat = 5/2

b_crits = compute_b_crits_SV(xi_t)
SV_kwargs = get_SV_kwargs(r_throat, xi_t)

rings_SV = find_rings_list(b_crits, SV_kwargs)
bs = compute_optimal_array_Npoints(0, 10, rings_SV, Npoints=50)
bs = bs[bs!=b_crits] # We remove b_crit because of singularity during integration
# bs = np.linspace(0,10,1000)

with warnings.catch_warnings(action="ignore"):
    plot_nturns(bs, b_crits, **SV_kwargs)

In [ ]:
r_throat = 5/2
xi_t = 1
SV_kwargs = get_SV_kwargs(r_throat, xi_t)

# b_crit = compute_b_crits_SV(xi_t)[0] # Note that compute_b_crits_SV returns a list
# bs = np.arange(0.1,10,0.1)

steps = (0.3,0.07,0.01)
inner_shadow = compute_inner_shadow(0,10,**SV_kwargs)
b_crits = compute_b_crits_SV(xi_t)
rings_SV = find_rings_list(b_crits, SV_kwargs)
bs_direct, bs_lensed, bs_p_ring = compute_optimal_array_steps(inner_shadow, 10, rings_SV, steps, joint=False)
bs = {
    'inner_shadow': ('black', np.arange(0, inner_shadow, steps[0])),
    'direct': ('dodgerblue', bs_direct),
    'lensed': ('orange', bs_lensed),
    'p_ring': ('red', bs_p_ring[~np.isin(bs_p_ring,b_crits)]), # We remove the value b_crit
}

make_geodesics_plot(bs, figsize=(my_width,my_width), savepath=None,
                    **SV_kwargs)

In [ ]:
steps = (0.3,0.07,0.01)
width = my_width*0.48*2/3
# width = my_width
savepath = None

# r_throats = [3/2,5/2]
# xi_ts = [-1,0,1]
r_throats = [3/2]
xi_ts = [-1]

for r_throat in r_throats:
    for xi_t in xi_ts:
        b_crits = compute_b_crits_SV(xi_t) # Note that compute_b_crits_SV returns a list
        SV_kwargs = get_SV_kwargs(r_throat, xi_t)
        rings_SV = find_rings_list(b_crits, SV_kwargs)

        inner_shadow = compute_inner_shadow(0,10,**SV_kwargs)
        bs_direct, bs_lensed, bs_p_ring = compute_optimal_array_steps(inner_shadow, 10, rings_SV, steps, joint=False)
        bs = {
            'inner_shadow': ('black', np.arange(0, inner_shadow, steps[0])),
            'direct': ('dodgerblue', bs_direct),
            'lensed': ('orange', bs_lensed),
            'p_ring': ('red', bs_p_ring[~np.isin(bs_p_ring,b_crits)]), # We remove the value b_crit
        }

        # savepath = f'latex/figures/SV/ray_tracing/RT_SV_r{int(r_throat)}_xi{round(xi_t)}.pdf'
        make_geodesics_plot(bs, figsize=(width,width), savepath=savepath,
                            **SV_kwargs)

### Transfer functions

In [ ]:
width = my_width*0.48*2/3

# r_throats = [3/2,5/2]
# xi_ts = [-1,0,1]
r_throats = [3/2]
xi_ts = [-1]

for r_throat in r_throats:
    for xi_t in xi_ts:
        b_crits = compute_b_crits_SV(xi_t)
        SV_kwargs = get_SV_kwargs(r_throat, xi_t)

        rings_SV = find_rings_list(b_crits, SV_kwargs)
        bs_list = compute_optimal_array_Npoints(0,10, rings_SV, Npoints=50, joint=False, fill=True)
        for bs in bs_list:
            bs = bs[~np.isin(bs,b_crits)]

        # savepath = f'latex/figures/SV/transfer_function/TF_SV_r{int(r_throat)}_xi{round(xi_t)}.pdf'

        make_transfer_function_plot(b_crits, SV_kwargs, bs_list,
                                    correction=0, figsize=(width,width), savepath=savepath)

### Emission models

Set the variable `emission_model` with one of the above defined emission profiles

In [ ]:
rs = np.linspace(0,13,1000)
width = my_width*0.48

# r_throats = [3/2,5/2]
# xi_ts = [-1,0,1.01]
r_throats = [3/2]
xi_ts = [-1]

for r_throat in r_throats:
    for xi_t in xi_ts:
        r_hor = compute_inner_edge_SV(r_throat)
        tick = r_hor
        ticklabel = r'$r_\mathrm{in}$'
        # tick = None
        # ticklabel = None

        params = (r_hor, 1/2, -2) # For small sigma it gives numerical error
        emission_model = lambda r: normalized_Standard_Unbound(r, *params)
        savepath = f'latex/figures/SV/shadows/Emitted_SV_r{int(r_throat)}_xi{round(xi_t)}.pdf'
        text_string = r'$r_{\mathrm{throat}}=$'+f' {r_throat}'+r'$M$'
        plot_emission_model(rs, emission_model, text_string, tick, ticklabel,
                            figsize=(width,width/golden), savepath=None)

In [ ]:
bs = np.linspace(0,10*np.sqrt(2),1000)
r_throat = 3/2
xi_t = 0
width = my_width*0.48

b_crits = compute_b_crits_SV(xi_t)
SV_kwargs = get_SV_kwargs(r_throat, xi_t)

rings_SV = find_rings_list(b_crits, SV_kwargs)
bs_transfer_list = compute_optimal_array_Npoints(0,10*np.sqrt(2), rings_SV, Npoints=100, joint=False, fill=True)

params = (SV_kwargs['inner_edge'], 1/2, -2) # For small sigma it gives numerical error
emission_model = lambda r: normalized_Standard_Unbound(r, *params)

# savepath = f'latex/figures/SV/shadows/Observed_SV_r{int(r_throat)}_xi{round(xi_t)}.pdf'
savepath = None
plot_observed_intensity(bs, bs_transfer_list, emission_model, SV_kwargs,
                        figsize=(width,width), savepath=savepath)

### Shadows

In [ ]:
width = my_width

# savepath = f'latex/figures/SV/shadows/Sh_SV_r{int(r_throat)}_xi{round(xi_t)}.pdf'
savepath = None
make_shadow_plot(bs_transfer_list, emission_model, SV_kwargs,
                 figsize=(width,width), savepath=savepath, y_range=None, Npixels=1.6e7)

In [ ]:
width = my_width*0.48
figsize = (width,width/golden)
savepath = None

rs = np.linspace(0,13,1000)

r_throats = [3/2,5/2]
# r_throats = [3/2]

for r_throat in r_throats:
    inner_edge = compute_inner_edge_SV(r_throat)
    tick = inner_edge
    ticklabel = r'$r_\mathrm{in}$'

    params = (inner_edge, 1/2, -2) # For small sigma it gives numerical error
    emission_model = lambda r: normalized_Standard_Unbound(r, *params)

    # savepath = f'latex/figures/SV/shadows/Emitted_SV_r{int(r_throat)}.pdf'
    text_string = r'$r_{\mathrm{throat}}=$'+f' {r_throat}'+r'$M$'
    plot_emission_model(rs, emission_model, text_string, tick, ticklabel,
                        figsize, savepath=savepath)

In [ ]:
y_range = (0,0.34)
width = my_width*0.48*2/3
figsize = (width,width)
savepath = None

bs = np.linspace(0,10*np.sqrt(2),1000)
rs = np.linspace(0,13,1000)

# r_throats = [3/2,5/2]
# xi_ts = [-1,0,1]
r_throats = [3/2]
xi_ts = [-1]

for r_throat in r_throats:
    for xi_t in xi_ts:
        b_crits = compute_b_crits_SV(xi_t)
        SV_kwargs = get_SV_kwargs(r_throat, xi_t)
        r_hor = SV_kwargs['inner_edge']

        rings_SV = find_rings_list(b_crits, SV_kwargs)
        bs_transfer_list = compute_optimal_array_Npoints(0,10*np.sqrt(2), rings_SV, Npoints=100, joint=False, fill=True)

        params = (r_hor, 1/2, -2) # For small sigma it gives numerical error
        emission_model = lambda r: normalized_Standard_Unbound(r, *params)
        
        # savepath = f'latex/figures/SV/shadows/Observed_SV_r{int(r_throat)}_xi{round(xi_t)}.pdf'
        plot_observed_intensity(bs, bs_transfer_list, emission_model, SV_kwargs,
                                figsize, savepath=savepath, y_range=y_range)
        
        # savepath = f'latex/figures/SV/shadows/Sh_SV_r{int(r_throat)}_xi{round(xi_t)}.pdf'
        figsize = (width/1.05,width/1.05)
        make_shadow_plot(bs_transfer_list, emission_model, SV_kwargs,
                         figsize, savepath=savepath, y_range=y_range)
        figsize = (width,width)


## Hayward

### Metric functions and potential

For the static case, the Hayward line element reads:
\begin{equation*}
    ds^2 = -e^{2\nu}dt^2+e^{2\lambda}dr^2+e^{2\xi}d\Omega^2
\end{equation*}
where
\begin{equation*}
    e^{2\nu}=e^{-2\lambda}=1-\frac{2Mr^2}{\left|r\right|^3 + 2M\gamma^2}
    \quad \mathrm{and} \quad
    e^{2\xi} = r^2+{r_{\mathrm{throat}}}^2.
\end{equation*}

Hereafter, we will lways work on units of $M=1$.

In [ ]:
def f_Hayward(r: float, gamma: float) -> float:
    """Radial function e^{2nu} of the Hayward metric in units of M=1"""
    return 1 - (2*r**2)/(np.abs(r)**3+2*gamma**2)

def areal_radius2(r: float, r_throat: float, xi_t: float) -> float:
    """areal radius squared e^{2xi} of a wormhole in units of M=1"""
    return ( np.sqrt(r**2+r_throat**2) + xi_t )**2

def compute_inner_edge_Hayward(gamma):
    """Inner edge for SV metric, for which g_tt(r)=0."""
    # NOTE: This is the exterior horizon
    gamma_crit = 4*np.sqrt(3)/9
    func = lambda x: 2/3 + (4/3)*np.cos( (1/3)*np.arccos(1-27*x**2/8) )
    return float(np.piecewise(gamma, [gamma <= gamma_crit , gamma > gamma_crit] , [func, 0] ))

def compute_photon_rings_Hayward(gamma, r_throat, xi_t):
    """Compute the coordinate r of the photon rings for the Hayward metric"""
    r_phs = []
    if gamma * areal_radius2(0, r_throat, xi_t) > 0:
        r_phs += [0] # There is a maximum at r=0
    neg_pot = lambda r: -potential(r, f_Hayward, (gamma,), areal_radius2, (r_throat,xi_t))
    result = optimize.minimize(neg_pot, 3) # Seed at r_ph for Schwarzschild
    local_max = result.x[0]
    if result.success and abs(local_max)>1e-4: # We ensure that both maxima don't coincide
        r_phs += [ local_max ] # We add the found local maximum
    inner_edge = compute_inner_edge_Hayward(gamma)
    r_phs = np.array(r_phs) # Cast to array to allow for boolean masks
    return r_phs[r_phs >= inner_edge] # Boolean mask for those that are not screened by the horizon

def compute_b_crits_Hayward(gamma, r_throat, xi_t):
    """Compute critical impact paremeters for Hayward spacetime"""
    r_phs = compute_photon_rings_Hayward(gamma, r_throat, xi_t)
    return 1./np.sqrt( potential(r_phs, f_Hayward, (gamma,), areal_radius2, (r_throat,xi_t)) )

def get_Hayward_kwargs(gamma, r_throat, xi_t):
    # TODO: Dar la posibilidad de que radial_fun y areal puedan no tener parámetros
    return {
    'r_phs': compute_photon_rings_Hayward(gamma, r_throat, xi_t),
    'inner_edge': compute_inner_edge_Hayward(gamma),
    'radial_fun': f_Hayward,
    'radial_params': (gamma,),
    'areal': areal_radius2,
    'areal_params': (r_throat, xi_t)
    }

In [ ]:
xi_ts = np.linspace(-0.575,-0.550,10)
r_throat = 1.5
gamma = 1

fun = lambda x: len(compute_b_crits_Hayward(gamma,r_throat,x))
a = -1
b = 0

while True:
    for i in np.linspace(a,b,100):
        if fun(i)==2:
            a = j
            b = i
            break
        j = i
    
    if abs(a-b)<1e-5:
        break

print(i)

First we take a look to the metric function $e^{2\nu}$:

In [ ]:
params = {r'$\gamma/M$' : [2, 1.5, 1, 4*np.sqrt(3)/9, 0.5, 0]}
r_range = np.linspace(0,10,200)
figsize = (my_width, my_width/golden)

with warnings.catch_warnings(action='ignore'):
    make_metric_function_plot(f_Hayward, params, r_range, figsize=figsize, savepath=None)

For the value $\gamma = 4\sqrt{3}/9\ M \approx 0.77M$, both horizons degenerate.

In [ ]:
def plot_potential_Hayward(ax, rr, gamma, xi_t, r_throat, label, linestyle, plot_text):
    kwargs_potential = {
        'radial_fun': f_Hayward,
        'radial_params': (gamma,),
        'areal': areal_radius2,
        'areal_params': (r_throat, xi_t)
    }
    ax.plot(rr, potential(rr, **kwargs_potential), label=label, linestyle=linestyle)
    if plot_text:
        ax.text(-2.5, 3, s=rf'$\gamma={gamma:.1f}M$')

def make_axins(ax, xlim, ylim, labelsize):
    axins = ax.inset_axes([0.50, 0.45, 0.45, 0.45])
    axins.set_xlim(xlim)
    axins.set_ylim(ylim)
    axins.tick_params(labelsize=labelsize)
    return axins

def plot_zoomed_potential(axins, rr, gamma, xi_t, r_throat, linestyle):
    plot_potential_Hayward(axins, rr, gamma, xi_t, r_throat, label=None, linestyle=linestyle, plot_text=False)
    return axins

def make_potential_plot_Hayward(rr, gammas, xi_ts, r_throat, linestyles,
                        figsize=(7,7), savepath=None, plot_text=True):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize, sharey=True)
    axins1 = make_axins(ax1, (0, 4.0), (0.0, 0.1), labelsize=8)
    ax1.indicate_inset_zoom(axins1, edgecolor="black", alpha=0.3)

    axins2 = make_axins(ax2, (0, 4.0), (0.0, 0.1), labelsize=8)
    ax2.indicate_inset_zoom(axins2, edgecolor="black", alpha=0.3)

    for i, xi_t in enumerate(xi_ts):
        label = r'$\Xi/M=$'+f' {xi_t}'
        plot_potential_Hayward(ax1, rr, gammas[0], xi_t, r_throat, label, linestyles[i], plot_text)
        axins1 = plot_zoomed_potential(axins1, rr, gammas[0], xi_t, r_throat, linestyles[i])

        plot_potential_Hayward(ax2, rr, gammas[1], xi_t, r_throat, None, linestyles[i], plot_text)
        axins2 = plot_zoomed_potential(axins2, rr, gammas[1], xi_t, r_throat, linestyles[i])

    ymax = max( [1./(r_throat+xi_t)**2 for xi_t in xi_ts] )
    ax1.set_ylabel(r'$V(r)/M^2$')
    for ax in [ax1,ax2]:
        ax.set_xlim( int(rr.min()) , int(rr.max()) )
        ax.set_ylim(0,ymax)
        ax.set_xlabel(r'$r/M$')

    handles, labels = ax1.get_legend_handles_labels()

    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.53, 1.1),
               ncol=4, frameon=True)
    plt.tight_layout()
    if savepath is not None:
        plt.savefig(savepath, format='pdf')
    plt.show()

In [ ]:
width = my_width
figsize = (width,width/3)
savepath=None
# savepath=f'latex/figures/Hayward/Hayward_potential.pdf'

rr = np.linspace(-3,7,200)
gammas = [0.5,1]
xi_ts = [-1,0,1]
r_throat = 3/2

linestyles = [ '--' , '-.' , '-' ]

make_potential_plot_Hayward(rr, gammas, xi_ts, r_throat, linestyles,
                            figsize=figsize, savepath=savepath, plot_text=True)

### Ray tracing

In [ ]:
gamma = 0
r_throat = 0
xi_t = 0
width = my_width*0.48

b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)
Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)

rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
bs = compute_optimal_array_Npoints(0.01, 100, rings_Hayward, Npoints=100)
# bs = np.linspace(0.01,10,100)
# bs = np.append( bs, np.linspace(b_crits[0]-1e-3, b_crits[0]+1e-3, 10000) )

plot_nturns(bs, b_crits, figsize=(width,width), **Hayward_kwargs)

In [ ]:
gamma = 1
xi_t = -1
r_throat = 3/2

Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)
# FIXME: no funciona para: gamma=1, xi_t=1, r_throat=3/2
rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)

steps = (0.3,0.1,0.1)
inner_shadow = compute_inner_shadow(0,10,**Hayward_kwargs)
bs_direct, bs_lensed, bs_p_ring = compute_optimal_array_steps(inner_shadow, 10, rings_Hayward, steps, joint=False)
bc1 = b_crits[0]
bs_inner = np.arange(bc1-1e-1, bc1+1e-1, 0.01)
bs_p_ring = np.append(bs_p_ring,bs_inner)
bs_p_ring = bs_p_ring[~np.isin(bs_p_ring,b_crits)]  # We remove the value b_crit
bs = {
    'inner_shadow': ('black', np.arange(0, inner_shadow, steps[1])),
    'direct': (None, bs_direct),
    'lensed': (None, bs_lensed),
    'p_ring': (None, bs_p_ring[~np.isin(bs_p_ring,b_crits)]),
}

# bs = compute_optimal_array_steps(0.2, 10, rings_Hayward, steps=(0.2,0.1,0.05))
# bs = bs[~np.isin(bs,b_crits)]  # type: ignore[index]

# bs = np.arange(0.1, 10, 0.1)

make_geodesics_plot(bs, figsize=(my_width,my_width), savepath=None,
                    **Hayward_kwargs)

In [ ]:
width = my_width*0.48*2/3
# width = my_width
savepath = None
r_throat = 3/2

##############
gamma = 0.5
xi_ts = [-1,0,1]

for xi_t in xi_ts:
    Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
    b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)
    rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
    
    steps = (0.3,0.07,0.01)
    inner_shadow = compute_inner_shadow(0,10,**Hayward_kwargs)
    bs_direct, bs_lensed, bs_p_ring = compute_optimal_array_steps(inner_shadow, 10, rings_Hayward, steps, joint=False)
    bs = {
        'inner_shadow': ('black', np.arange(0, inner_shadow, steps[0])),
        'direct': ('dodgerblue', bs_direct),
        'lensed': ('orange', bs_lensed),
        'p_ring': ('red', bs_p_ring[~np.isin(bs_p_ring,b_crits)]), # We remove the value b_crit
    }

    # savepath = f'latex/figures/Hayward/ray_tracing/RT_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
    make_geodesics_plot(bs, figsize=(width,width), savepath=savepath,
                        **Hayward_kwargs)

##############
gamma = 1
xi_t = -1

Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)
rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)

steps = (0.3,0.1,0.1)
inner_shadow = compute_inner_shadow(0,10,**Hayward_kwargs)
bs_direct, bs_lensed, bs_p_ring = compute_optimal_array_steps(inner_shadow, 10, rings_Hayward, steps, joint=False)
bc1 = b_crits[0]
bs_inner = np.arange(bc1-1e-1, bc1+1e-1, 0.01)
bs_p_ring = np.append(bs_p_ring,bs_inner)
bs_p_ring = bs_p_ring[~np.isin(bs_p_ring,b_crits)]  # We remove the value b_crit
bs = {
    'inner_shadow': ('black', np.arange(0, inner_shadow, steps[1])),
    'direct': (None, bs_direct),
    'lensed': (None, bs_lensed),
    'p_ring': (None, bs_p_ring[~np.isin(bs_p_ring,b_crits)]),
}

# savepath = f'latex/figures/Hayward/ray_tracing/RT_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
make_geodesics_plot(bs, figsize=(width,width), savepath=savepath,
                    **Hayward_kwargs)

##############
gamma = 1
xi_t = 0

Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)
rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)

steps = (0.3,0.1,0.1)
inner_shadow = compute_inner_shadow(0,10,**Hayward_kwargs)
bs_direct, bs_lensed, bs_p_ring = compute_optimal_array_steps(inner_shadow, 10, rings_Hayward, steps, joint=False)
bs_p_ring = bs_p_ring[~np.isin(bs_p_ring,b_crits)]  # We remove the value b_crit
bs = {
    'inner_shadow': ('black', np.arange(0, inner_shadow, steps[0])),
    'direct': (None, bs_direct),
    'lensed': (None, bs_lensed),
    'p_ring': (None, bs_p_ring[~np.isin(bs_p_ring,b_crits)]),
}

# savepath = f'latex/figures/Hayward/ray_tracing/RT_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
make_geodesics_plot(bs, figsize=(width,width), savepath=savepath,
                    **Hayward_kwargs)
    
# ##############
gamma = 1
xi_t = 1

Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)

# We manually add bs for the inner critical impact parameter (bc1)
bc1 = b_crits[0]
bs_inner_p_ring = np.arange(bc1-1e-7, bc1+1e-6, 2e-7)
bs_inner_lensed = np.arange(bc1-1e-3, bc1+8e-3, 2e-3)
bs_inner = np.concat([bs_inner_lensed,bs_inner_p_ring])

# bs for the outer one
b_crits = b_crits[1:]

rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)

steps = (0.3,0.1,0.1)
inner_shadow = compute_inner_shadow(0,10,**Hayward_kwargs)
bs_direct, bs_lensed, bs_p_ring = compute_optimal_array_steps(inner_shadow, 10, rings_Hayward, steps, joint=False)
bs_p_ring = np.append(bs_p_ring,bs_inner)
bs_p_ring = bs_p_ring[~np.isin(bs_p_ring,b_crits)]  # We remove the value b_crit
bs = {
    'inner_shadow': ('black', np.arange(0, inner_shadow, steps[0])),
    'direct': (None, bs_direct),
    'lensed': (None, bs_lensed),
    'p_ring': (None, bs_p_ring[~np.isin(bs_p_ring,b_crits)]),
}

# savepath = f'latex/figures/Hayward/ray_tracing/RT_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
make_geodesics_plot(bs, figsize=(width,width), savepath=savepath,
                    **Hayward_kwargs)

In [ ]:
gamma = 1
r_throat = 1.5
xi_t = -1
Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)
rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)

In [ ]:
rings_Hayward

### Transfer functions

In [ ]:
width = my_width*0.48*2/3
savepath = None

gamma = 0.5
xi_t = 1
r_throat = 3/2

Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)

# FIXME: para gamma=1, xi_t=-1, r_throat=3/2 hay solo una photon sphere pero hay strong lensing fuera
# FIXME: no funciona para gamma=1, xi_t=1, r_throat=3/2. No encuentra photon ring edge
rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
bs_list = compute_optimal_array_Npoints(0, 10, rings_Hayward,
                                        Npoints=100, joint=False, fill=True)

# savepath = f'latex/figures/Hayward/transfer_function/TF_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
make_transfer_function_plot(b_crits, Hayward_kwargs, bs_list,
                                    figsize=(width,width), savepath=savepath)

In [ ]:
width = my_width*0.48*2/3
savepath = None
r_throat = 3/2

##############
gamma = 0.5
xi_ts = [-1,0,1]
for xi_t in xi_ts:
    Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
    b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)

    rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
    bs_list = compute_optimal_array_Npoints(0, 10, rings_Hayward,
                                            Npoints=100, joint=False, fill=True)

    # savepath = f'latex/figures/Hayward/transfer_function/TF_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
    make_transfer_function_plot(b_crits, Hayward_kwargs, bs_list,
                                figsize=(width,width), savepath=savepath)
    
##############
gamma = 1
xi_ts = [-1, 0]
for xi_t in xi_ts:
    Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
    b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)

    rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
    Npoints = 100 if xi_t==0 else 500
    bs_list = compute_optimal_array_Npoints(0, 10, rings_Hayward,
                                            Npoints=Npoints, joint=False, fill=True)

    # savepath = f'latex/figures/Hayward/transfer_function/TF_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
    make_transfer_function_plot(b_crits, Hayward_kwargs, bs_list,
                                figsize=(width,width), savepath=savepath)

In [ ]:
# Celda separada porque cuando se hace en la anterior con el resto da error (???)
width = my_width*0.48*2/3
savepath = None

r_throat = 3/2
gamma = 1
xi_t = 1

Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)

# We manually add bs for the inner critical impact parameter (bc1)
bc1 = b_crits[0]
bs_inner_lensed = np.linspace(bc1-1e-2, bc1+1e-2, 100)
bs_inner_p_ring = np.linspace(bc1-1e-4, bc1+1e-4, 3000)

# bs for the outer one
b_crits = b_crits[1:]
rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
[bs_direct, bs_lensed, bs_p_ring] = compute_optimal_array_Npoints(0, 10, rings_Hayward,
                                                                Npoints=100, joint=False, fill=True)
bs_list = [ bs_direct, [bs_inner_lensed, bs_lensed], [bs_inner_p_ring, bs_p_ring] ]

# savepath = f'latex/figures/Hayward/transfer_function/TF_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
make_transfer_function_plot(b_crits, Hayward_kwargs, bs_list,
                            figsize=(width,width), savepath=savepath)

### Emission models

In [ ]:
width = my_width*0.48
figsize = (width,width/golden)
savepath = None

rs = np.linspace(0,13,1000)
gammas = [0.5, 1]

for gamma in gammas:
    inner_edge = compute_inner_edge_Hayward(gamma)
    tick = inner_edge
    ticklabel = r'$r_\mathrm{in}$'

    params = (inner_edge, 1/2, -2) # For small sigma it gives numerical error
    emission_model = lambda r: normalized_Standard_Unbound(r, *params)
    # emission_model = lambda r: intensity_at_inner_edge(r, inner_edge)

    # savepath = f'latex/figures/Hayward/shadows/Emitted_Hayward_g{int(gamma)}.pdf'
    text_string = rf'$\gamma={gamma:.1f}M$'
    plot_emission_model(rs, emission_model, text_string, tick, ticklabel,
                        figsize, savepath=savepath)

In [ ]:
width = my_width*0.48
figsize = (width,width/golden)
savepath = None
savepath = f'latex/figures/Emission_models.pdf'

rs = np.linspace(0,13,1000)

peaks = {
    'SV BH': compute_inner_edge_SV(3/2), 
    'Hayward BH': compute_inner_edge_Hayward(0.5),
    'Wormholes': 0
}

fig, ax = plt.subplots(figsize=figsize)
for label, mu in peaks.items():
    params = (mu, 1/2, -2) # For small sigma it gives numerical error
    emission_model = lambda r: normalized_Standard_Unbound(r, *params)
    ax.plot(rs, emission_model(rs), label=label)

rmin = round(rs.min())
rmax = round(rs.max())

ax.set_xlabel(r'$r/M$')
ax.set_xlim(rmin,rmax)
ax.set_xticks(np.arange(0,13,2))
ax.set_ylabel(r'$I_\mathrm{em}/I_0$')
ax.set_ylim(0,1)
ax.legend()
if savepath is not None:
    plt.savefig(savepath, format='pdf')
plt.show()

In [ ]:
width = my_width*0.48*2/3
savepath = None
r_throat = 3/2

##############
gamma = 0.5
xi_ts = [-1,0,1]
for xi_t in xi_ts:
    Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
    b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)

    rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
    bs_transfer_list = compute_optimal_array_Npoints(0, 10*np.sqrt(2), rings_Hayward,
                                                     Npoints=100, joint=False, fill=True)

    params = (Hayward_kwargs['inner_edge'], 1/2, -2) # For small sigma it gives numerical error
    emission_model = lambda r: normalized_Standard_Unbound(r, *params)

    # savepath = f'latex/figures/Hayward/shadows/Observed_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
    bs = np.linspace(0,10*np.sqrt(2),1000)
    plot_observed_intensity(bs, bs_transfer_list, emission_model, Hayward_kwargs,
                            figsize=(width,width), savepath=savepath)
    
##############
gamma = 1
xi_ts = [-1,0]
for xi_t in xi_ts:
    Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
    b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)

    rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
    Npoints = 100 if xi_t==0 else 500
    bs_transfer_list = compute_optimal_array_Npoints(0, 10*np.sqrt(2), rings_Hayward,
                                                     Npoints=Npoints, joint=False, fill=True)

    params = (Hayward_kwargs['inner_edge'], 1/2, -2) # For small sigma it gives numerical error
    emission_model = lambda r: normalized_Standard_Unbound(r, *params)

    # savepath = f'latex/figures/Hayward/shadows/Observed_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
    bs = np.linspace(0,10*np.sqrt(2),1000)
    plot_observed_intensity(bs, bs_transfer_list, emission_model, Hayward_kwargs,
                            figsize=(width,width), savepath=savepath)

In [ ]:
gamma = 1
xi_t = 1

Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)

# We manually add bs for the inner critical impact parameter (bc1)
bc1 = b_crits[0]
bs_inner_lensed = np.linspace(bc1-1e-2, bc1+1e-2, 100)
bs_inner_p_ring = np.linspace(bc1-1e-4, bc1+1e-4, 100)

# bs for the outer one
b_crits = b_crits[1:]
rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
[bs_direct, bs_lensed, bs_p_ring] = compute_optimal_array_Npoints(0, 10*np.sqrt(2), rings_Hayward,
                                                                  Npoints=100, joint=False, fill=True)
bs_lensed = np.sort( np.append(bs_lensed,bs_inner_lensed) )
bs_p_ring = np.sort( np.append(bs_p_ring,bs_inner_p_ring) )
bs_transfer_list = [ bs_direct, bs_lensed, bs_p_ring ]

params = (Hayward_kwargs['inner_edge'], 1/2, -2) # For small sigma it gives numerical error
emission_model = lambda r: normalized_Standard_Unbound(r, *params)

# savepath = f'latex/figures/Hayward/shadows/Observed_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
bs = np.linspace(0,10*np.sqrt(2),1000)
bs = np.append(bs, bs_inner_lensed)
bs = bs[bs!=bc1]
bs.sort()
plot_observed_intensity(bs, bs_transfer_list, emission_model, Hayward_kwargs,
                        figsize=(width,width), savepath=savepath)

### Shadows

In [ ]:
width = my_width*0.48*2/3
savepath = None

gamma = 0.5
xi_t = 0
r_throat = 3/2

Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)

rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
bs_transfer_list = compute_optimal_array_Npoints(0, 10*np.sqrt(2), rings_Hayward,
                                                 Npoints=100, joint=False, fill=True)

params = (Hayward_kwargs['inner_edge'], 1/2, -2) # For small sigma it gives numerical error
emission_model = lambda r: normalized_Standard_Unbound(r, *params)

# savepath = f'latex/figures/Hayward/shadows/Sh_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
make_shadow_plot(bs_transfer_list, emission_model, Hayward_kwargs,
                 figsize=(width,width), savepath=savepath, y_range=None, Npixels=1.6e7)

In [ ]:
width = my_width*0.48*2/3
savepath = None
r_throat = 3/2

##############
gamma = 0.5
xi_ts = [-1,0,1]
for xi_t in xi_ts:
    Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
    b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)

    rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
    bs_transfer_list = compute_optimal_array_Npoints(0, 10*np.sqrt(2), rings_Hayward,
                                                    Npoints=100, joint=False, fill=True)

    params = (Hayward_kwargs['inner_edge'], 1/2, -2) # For small sigma it gives numerical error
    emission_model = lambda r: normalized_Standard_Unbound(r, *params)

    # savepath = f'latex/figures/Hayward/shadows/Sh_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
    make_shadow_plot(bs_transfer_list, emission_model, Hayward_kwargs,
                    figsize=(width,width), savepath=savepath, y_range=None, Npixels=1.6e7)
    
##############
gamma = 1
xi_ts = [-1,0]
for xi_t in xi_ts:
    Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
    b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)

    rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
    bs_transfer_list = compute_optimal_array_Npoints(0, 10*np.sqrt(2), rings_Hayward,
                                                    Npoints=100, joint=False, fill=True)

    params = (Hayward_kwargs['inner_edge'], 1/2, -2) # For small sigma it gives numerical error
    emission_model = lambda r: normalized_Standard_Unbound(r, *params)

    # savepath = f'latex/figures/Hayward/shadows/Sh_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
    make_shadow_plot(bs_transfer_list, emission_model, Hayward_kwargs,
                    figsize=(width,width), savepath=savepath, y_range=None, Npixels=1.6e7)

In [ ]:
# Aquí también en celda separada porque cuando se hace en la anterior con el resto da error
width = my_width*0.48*2/3
savepath = None
r_throat = 3/2
gamma = 1
xi_t = 1

Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)

# We manually add bs for the inner critical impact parameter (bc1)
bc1 = b_crits[0]
bs_inner_lensed = np.linspace(bc1-1e-2, bc1+1e-2, 100)
bs_inner_p_ring = np.linspace(bc1-1e-4, bc1+1e-4, 1000)

# bs for the outer one
b_crits = b_crits[1:]
rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
[bs_direct, bs_lensed, bs_p_ring] = compute_optimal_array_Npoints(0, 10*np.sqrt(2), rings_Hayward,
                                                                  Npoints=100, joint=False, fill=True)
bs_lensed = np.sort( np.append(bs_lensed,bs_inner_lensed) )
bs_p_ring = np.sort( np.append(bs_p_ring,bs_inner_p_ring) )
bs_transfer_list = [ bs_direct, bs_lensed, bs_p_ring ]

params = (Hayward_kwargs['inner_edge'], 1/2, -2) # For small sigma it gives numerical error
emission_model = lambda r: normalized_Standard_Unbound(r, *params)

# savepath = f'latex/figures/Hayward/shadows/Sh_Hayward_g{int(gamma)}_xi{round(xi_t)}.pdf'
make_shadow_plot(bs_transfer_list, emission_model, Hayward_kwargs,
                figsize=(width,width), savepath=savepath, y_range=None, Npixels=1.6e7)

## Schwarzschild

In [ ]:
def f_Schwarzschid(r: float, dummy) -> float:
    """Radial function e^{2nu} of the Schwarzschild metric in units of M=1"""
    return 1 - 2./r

def areal_radius2(r: float, dummy) -> float:
    """areal radius squared e^{2xi} of a wormhole in units of M=1"""
    return r**2

def compute_inner_edge_Schwarzschid():
    """Inner edge for Schwarzschild metric, for which g_tt(r)=0"""
    return 2.

def compute_photon_rings_Schwarzschid():
    """Compute the coordinate r of the photon ring for the Schwarzschid metric"""
    return [3.]

def compute_b_crits_Schwarzschid():
    """Compute the critical impact paremeter of a Schwarzschid black hole"""
    return [np.sqrt(27)]

def get_Schwarzschid_kwargs():
    # TODO: Dar la posibilidad de que radial_fun y areal puedan no tener parámetros
    return {
    'r_phs': compute_photon_rings_Schwarzschid(),
    'inner_edge': compute_inner_edge_Schwarzschid(),
    'radial_fun': f_Schwarzschid,
    'radial_params': None,
    'areal': areal_radius2,
    'areal_params': None,
    }

In [ ]:
b_crits = compute_b_crits_Schwarzschid()
Schwarzschild_kwargs = get_Schwarzschid_kwargs()
rings_Schwarzschild = find_rings_list(b_crits, Schwarzschild_kwargs)

width = my_width # Figsize

In [ ]:
bs = compute_optimal_array_Npoints(0, 10, rings_Schwarzschild, Npoints=50)
bs = bs[bs!=b_crits] # We remove b_crit because of singularity during integration

with warnings.catch_warnings(action="ignore"):
    plot_nturns(bs, b_crits, **Schwarzschild_kwargs)

In [ ]:
# bs = compute_optimal_array_steps(0, 10, rings_Schwarzschild)
# bs = bs[bs!=b_crits] # We remove b_crit because of singularity during integration

# bs_dict = np.arange(0.1,10,0.1)

steps = (0.2, 0.04, 0.008)
inner_shadow = compute_inner_shadow(0,10,**Schwarzschild_kwargs)
bs_colorful = {
    'direct': ('green', np.arange(rings_Schwarzschild[0]['lensed'][1], 10, steps[0])),
    'lensed': ('orange', np.arange(*rings_Schwarzschild[0]['lensed'], steps[1])),
    'p_ring': ('red', np.arange(*rings_Schwarzschild[0]['p_ring'], steps[2])[1:]), # We remove the value b_crit
    'retro_p_ring': ('blue', np.arange(*rings_Schwarzschild[0]['retro_p_ring'], steps[2])),
    'retro_lensed': ('purple', np.arange(*rings_Schwarzschild[0]['retro_lensed'], steps[1])),
    'retro_direct': ('cyan', np.arange(inner_shadow, rings_Schwarzschild[0]['retro_lensed'][0], steps[0])),
    'inner_shadow': ('black', np.arange(0, inner_shadow, steps[0]))
}

make_geodesics_plot(bs_colorful, figsize=(width,width), **Schwarzschild_kwargs)

In [ ]:
# A different color choice

steps = (0.2, 0.04, 0.004)
inner_shadow = compute_inner_shadow(0,10,**Schwarzschild_kwargs)
bs_direct, bs_lensed, bs_p_ring = compute_optimal_array_steps(inner_shadow, 10, rings_Schwarzschild, steps, joint=False)
bs_colorful = {
    'direct': ('blue', bs_direct),
    'lensed': ('orange', bs_lensed),
    'p_ring': ('red', bs_p_ring[~np.isin(bs_p_ring,b_crits)]), # We remove the value b_crit
    'inner_shadow': ('black', np.arange(0, inner_shadow, steps[0]))
}

make_geodesics_plot(bs_colorful, figsize=(width,width), **Schwarzschild_kwargs)

In [ ]:
bs_list = compute_optimal_array_Npoints(0,10, rings_Schwarzschild, Npoints=100, joint=False, fill=True)

make_transfer_function_plot(b_crits, Schwarzschild_kwargs, bs_list,
                            correction=0, figsize=(width,width))

In [ ]:
rs = np.linspace(0,13,1000)

r_hor = compute_inner_edge_Schwarzschid()
tick = r_hor
ticklabel = r'$r_\mathrm{hor}$'

params = (r_hor, 1/2, -2) # For small sigma it gives numerical error
emission_model = lambda r: normalized_Standard_Unbound(r, *params)
plot_emission_model(rs, emission_model, tick=tick, ticklabel=ticklabel,
                    figsize=(width,width/golden), savepath=None)

In [ ]:
bs = np.linspace(0,10*np.sqrt(2),1000)
bs_transfer_list = compute_optimal_array_Npoints(0,10*np.sqrt(2), rings_Schwarzschild, Npoints=100, joint=False, fill=True)
plot_observed_intensity(bs, bs_transfer_list, emission_model, Schwarzschild_kwargs, figsize=(width,width))

In [ ]:
make_shadow_plot(bs_transfer_list, emission_model, Schwarzschild_kwargs, figsize=(width,width))

## Strong lensing

In [ ]:
def f_Hayward(r: float, gamma: float) -> float:
    """Radial function e^{2nu} of the Hayward metric in units of M=1"""
    return 1 - (2*r**2)/(np.abs(r)**3+2*gamma**2)

def areal_radius2(r: float, r_throat: float, xi_t: float) -> float:
    """areal radius squared e^{2xi} of a wormhole in units of M=1"""
    return ( np.sqrt(r**2+r_throat**2) + xi_t )**2

def compute_inner_edge_Hayward(gamma):
    """Inner edge for SV metric, for which g_tt(r)=0."""
    # NOTE: This is the exterior horizon
    gamma_crit = 4*np.sqrt(3)/9
    func = lambda x: 2/3 + (4/3)*np.cos( (1/3)*np.arccos(1-27*x**2/8) )
    
    return float(np.piecewise(gamma, [gamma <= gamma_crit , gamma > gamma_crit] , [func, 0] ))

def compute_photon_rings_Hayward(gamma, r_throat, xi_t):
    """Compute the coordinate r of the photon rings for the Hayward metric"""
    r_phs = []
    if gamma * areal_radius2(0, r_throat, xi_t) > 0:
        r_phs += [0] # There is a maximum at r=0
    neg_pot = lambda r: -potential(r, f_Hayward, (gamma,), areal_radius2, (r_throat,xi_t))
    result = optimize.minimize(neg_pot, 3) # Seed at r_ph for Schwarzschild
    local_max = result.x[0]
    if result.success and abs(local_max)>1e-4: # We ensure that both maxima don't coincide
        r_phs += [ local_max ] # We add the found local maximum
    inner_edge = compute_inner_edge_Hayward(gamma)
    r_phs = np.array(r_phs) # Cast to array to allow for boolean masks
    return r_phs[r_phs >= inner_edge] # Boolean mask for those that are not screened by the horizon

def compute_b_crits_Hayward(gamma, r_throat, xi_t):
    """Compute critical impact paremeters for Hayward spacetime"""
    r_phs = compute_photon_rings_Hayward(gamma, r_throat, xi_t)
    return 1./np.sqrt( potential(r_phs, f_Hayward, (gamma,), areal_radius2, (r_throat,xi_t)) )

def get_Hayward_kwargs(gamma, r_throat, xi_t):
    # TODO: Dar la posibilidad de que radial_fun y areal puedan no tener parámetros
    return {
    'r_phs': compute_photon_rings_Hayward(gamma, r_throat, xi_t),
    'inner_edge': compute_inner_edge_Hayward(gamma),
    'radial_fun': f_Hayward,
    'radial_params': (gamma,),
    'areal': areal_radius2,
    'areal_params': (r_throat, xi_t)
    }

In [ ]:
def plot_potential_Hayward(ax, rr, gamma, xi_t, r_throat, label=None, color=None, linestyle=None):
    kwargs_SV = {
        'radial_fun': f_Hayward,
        'radial_params': (gamma,),
        'areal': areal_radius2,
        'areal_params': (r_throat, xi_t)
    }
    ax.plot(rr, potential(rr, **kwargs_SV), label=label, color=color, linestyle=linestyle, linewidth=1)

In [ ]:
width = my_width*0.48
figsize = (width,width)
# savepath=f'latex/figures/Hayward/Hayward_potential.pdf'
savepath=None

rr = np.linspace(0.1,4,200)
gammas = [0, (25/24)*np.sqrt(5/6), 1.03, 1.16, 2.15]
xi_t = 0
r_throat = 0


fig, ax = plt.subplots(figsize=figsize)

for gamma in gammas[:1]:
    plot_potential_Hayward(ax, rr, gamma, xi_t, r_throat, color='black', linestyle=(0,(5,5)))
for gamma in gammas[1:2]:
    plot_potential_Hayward(ax, rr, gamma, xi_t, r_throat, rf'$\gamma={gamma:.2f}...$')
    b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)
    Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
    r_ph = Hayward_kwargs['r_phs'][0]
    pot_max = 1/b_crits[0]**2
    ax.vlines(r_ph, 0, pot_max, colors='gray',linestyles=(0,(1,3)),linewidth=1,zorder=0)
    ax.hlines(pot_max, 0, r_ph, colors='gray',linestyles=(0,(1,3)),linewidth=1,zorder=0)
for gamma in gammas[2:]:
    plot_potential_Hayward(ax, rr, gamma, xi_t, r_throat, rf'$\gamma={gamma:.2f}$')

ax.set_ylabel(r'$V(r)/M^2$')

ax.set_xlim( int(rr.min()) , int(rr.max()) )
ax.set_ylim(0,0.2)
ax.set_xlabel(r'$r/M$')
ax.legend()

plt.show()

In [ ]:
width = my_width*0.48
figsize = (width,width)

r_throat = 0
xi_t = 0

ax = plt.figure(figsize=figsize).add_subplot()
xticks = [i for i in range(0,11,2)]
xtick_labels = [f'{tick}' for tick in xticks]

# Schwarzschild
gamma = 0
b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)
Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
bs_Schwarzschild = compute_optimal_array_Npoints(0.01, 100, rings_Hayward, Npoints=100)
ax.plot(bs_Schwarzschild, compute_nturns(bs_Schwarzschild, **Hayward_kwargs),
        color='black', linestyle=(0,(5,5)), linewidth=1)

# # Marginally unstable photon sphere
gamma = (25/24)*np.sqrt(5/6)  
b_crits = compute_b_crits_Hayward(gamma, r_throat, xi_t)
Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
rings_Hayward = find_rings_list(b_crits, Hayward_kwargs)
bs_marginal = compute_optimal_array_Npoints(0.01, 100, rings_Hayward, Npoints=100)
ax.plot(bs_marginal, compute_nturns(bs_marginal, **Hayward_kwargs), 
        label=rf'$\gamma={gamma:.2f}...$', linewidth=1)
ax.vlines(b_crits,0,2,colors='gray',linestyles=(0,(1,3)),linewidth=1,zorder=0)
xticks += list(b_crits)
xtick_labels += [r'$b_c$']

# Strong lensing
bs = np.linspace(0.01,10,500)
gammas = [1.03, 1.16, 2.15]
for gamma in gammas:
    Hayward_kwargs = get_Hayward_kwargs(gamma, r_throat, xi_t)
    ax.plot(bs, compute_nturns(bs, **Hayward_kwargs),
            label=rf'$\gamma={gamma:.2f}$', linewidth=1)

ax.hlines([0.75+i*0.5 for i in range(3)],0,10,colors='gray',linestyles='solid',zorder=0, linewidth=0.5)

ax.set_xlim(0,10)
ax.set_ylim(0,2)
ax.set_xticks(xticks)
ax.set_xticklabels(xtick_labels)
ax.set_yticks(np.arange(0,2.1,0.25))
ax.set_xlabel(r'$b/M$')
ax.set_ylabel(r'$n=\phi /(2\pi)$')
ax.legend(loc='upper right',framealpha=1)

plt.show()